# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 25

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.

The four main types of exercise are aerobic (cardio), strength training, flexibility, and balance exercises. A well-rounded fitness routine includes all four types. Adults should aim for at least 150 minutes of moderate-intensity aerobic activity per week, along with muscle-strengthening activities on 2 or more days per week.

Chapter 2: Exercises for Common Problems' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 3})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o", max_tokens=1000,temperature=0)

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Partial Crunches**: Lie on your back with knees bent, cross arms over your chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises involve gentle stretching and strengthening, which

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health by playing a crucial role in physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, is essential for these processes. Sleep occurs in cycles, including REM and non-REM stages, each contributing to different aspects of health, such as memory, learning, and physical repair. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are important for promoting consistent, quality sleep and, consequently, overall health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'For stress relief, some natural remedies include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4.\n- Progressive muscle relaxation: Tense and release muscle groups from toes to head.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, 1 you taste.\n- Taking a short walk, preferably in nature.\n- Listening to calming music.\n\nFor managing headaches naturally, you can try:\n\n- Drinking water to stay hydrated.\n- Applying a cold or warm compress to your head or neck.\n- Resting in a dark, quiet room.\n- Gently massaging your temples and neck.\n- Using peppermint or lavender essential oils.\n- Consuming caffeine in small amounts, as it can help or hurt.\n- Maintaining a regular sleep schedule.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"I don't know. The provided context does not contain information on how sleep affects overall health."

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches, based on the context provided, include:\n\n1. **Relaxation Techniques**: Engaging in relaxation exercises such as progressive muscle relaxation, meditation, and deep breathing exercises can help manage stress and alleviate headaches.\n\n2. **Herbal Teas**: Drinking herbal teas like chamomile or valerian root may promote relaxation and help reduce stress.\n\n3. **Light Stretching or Yoga**: Incorporating light stretching or yoga into your routine can help relieve muscle tension and reduce stress, which may also alleviate headaches.\n\n4. **Warm Bath or Shower**: Taking a warm bath or shower can be a soothing way to relax and reduce stress, potentially easing headache symptoms.\n\n5. **Journaling or Gratitude Practice**: Engaging in journaling or a gratitude practice can help manage stress by providing an outlet for emotions and promoting a positive mindset.\n\nThese remedies focus on both immediate relief and long-term lifestyle changes t

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

*Your answer here*
BM25 is best applied in cases where we need to retrieve a rare or exact token from a document. It performs particularly well when the goal is to match an exact string, such as a specific error code.
For example:
“Error 00987GTY90 when updating Mac update”
Here, 00987GTY90 is a unique error identifier. If this exact string appears in a document, BM25 will strongly prioritize that document because it relies on exact term matching and assigns high importance to rare tokens.
In contrast, embedding-based models may treat 00987GTY90 as a meaningless or fragmented token. As a result, they might return documents about general Mac update issues instead of targeting that specific error code.
Therefore, BM25 excels when queries contain structured or standardized identifiers such as:
Error codes, Legal case numbers, Product IDs, Version numbers.

In these cases, exact lexical matching is more important than semantic similarity. Embedding-based models tend to generalize, whereas BM25 preserves and prioritizes unique identifiers.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is crucial for overall health as it plays a vital role in physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, is essential for these processes. Sleep occurs in cycles, including REM and non-REM stages, each contributing to different aspects of health, such as memory, learning, and physical repair. Therefore, maintaining good sleep hygiene and a consistent sleep schedule is important for promoting overall health.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\nFor headaches:\n- Drink water and stay hydrated\n- Apply a cold or warm compress to the head or neck\n- Rest in a dark, quiet room\n- Gentle massage of temples and neck\n- Use peppermint or lavender essential oils\n- Consume caffeine in small amounts (can help or hurt)\n- Maintain a regular sleep schedule\n\nFor stress:\n- Practice deep breathing exercises\n- Engage in progressive muscle relaxation\n- Use grounding techniques\n- Take a short walk, preferably in nature\n- Listen to calming music\n\nFor long-term stress management:\n- Regular exercise\n- Adequate sleep\n- Maintain social connections and support\n- Practice time management and prioritization\n- Set healthy boundaries\n- Engage in hobbies and leisure activities\n- Limit news and social media consumption'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises involve gentle stretching and strengthening, which can help alleviate 

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, is essential for these processes. Sleep occurs in cycles, including REM and non-REM stages, each playing a role in maintaining health. For example, deep sleep is important for body repair and regeneration, while REM sleep is vital for memory and learning. Therefore, maintaining good sleep hygiene and ensuring quality sleep can greatly benefit overall health.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\nFor headaches:\n- Drink water and stay hydrated to prevent dehydration-related headaches.\n- Apply a cold or warm compress to your head or neck.\n- Rest in a dark, quiet room to alleviate symptoms.\n- Gently massage your temples and neck.\n- Use peppermint or lavender essential oils for their soothing properties.\n- Consume caffeine in small amounts, as it can help some people.\n\nFor stress:\n- Practice deep breathing exercises, such as inhaling for 4 counts, holding for 4, and exhaling for 4.\n- Try progressive muscle relaxation by tensing and releasing muscle groups from toes to head.\n- Use grounding techniques, like naming 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Take a short walk, preferably in nature, to clear your mind.\n- Listen to calming music to help relax.\n\nFor long-term stress management:\n- Engage in regular exercise and ensure adequate sleep.\n- Maintain social connections an

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

*Your answer here*
Recall measures "How many of the relevant documents in the collection are successfully retrieved?". If relevant documents exist but your system fails to retrieve them, recall is low. What Query Reformulation Does is instead of using just one query, the system generates multiple variations. Those variations captures different vocabulary variants of the same context. And then when we search all of them we retrieve all of them. So more relevant documents are retrieved - which means higher recall. This works because different documents use different terminology. Example : 100 relevant documents exist. Original query retrieves 60 → Recall = 60%. With 5 reformulations, we retrieve 85 → Recall = 85%. With reformulations we increased the coverage of relevant language. Reformulations cast a wider net with semantic features.



## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and sleep occurs in cycles of about 90 minutes, alternating between REM (rapid eye movement) and non-REM sleep. Proper sleep hygiene, such as maintaining a consistent sleep schedule and creating a relaxing bedtime routine, can improve sleep quality and, consequently, overall health.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\nFor stress:\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4.\n- Progressive muscle relaxation: Tense and release muscle groups from toes to head.\n- Grounding techniques: Engage your senses by naming things you see, hear, feel, smell, and taste.\n- Taking a short walk, preferably in nature.\n- Listening to calming music.\n- Regular exercise and adequate sleep.\n- Mindfulness and meditation practices.\n\nFor headaches:\n- Drink water and stay hydrated.\n- Apply a cold or warm compress to the head or neck.\n- Rest in a dark, quiet room.\n- Gentle massage of temples and neck.\n- Use peppermint or lavender essential oils.\n- Maintain a regular sleep schedule.\n\nThese remedies can help manage stress and headaches naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Partial Crunches**: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and sleep occurs in cycles of about 90 minutes, alternating between REM (rapid eye movement) and non-REM sleep. Proper sleep hygiene, such as maintaining a consistent sleep schedule and creating a relaxing bedtime routine, can improve sleep quality and, consequently, overall health.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\nFor stress:\n- Immediate relief techniques such as deep breathing, progressive muscle relaxation, grounding techniques, taking a short walk (preferably in nature), and listening to calming music.\n- Long-term stress management strategies like regular exercise, adequate sleep, maintaining social connections and support, effective time management, setting healthy boundaries, engaging in hobbies and leisure activities, and limiting news and social media consumption.\n\nFor headaches:\n- Staying hydrated by drinking water.\n- Applying a cold or warm compress to the head or neck.\n- Resting in a dark, quiet room.\n- Gently massaging the temples and neck.\n- Using peppermint or lavender essential oils.\n- Consuming caffeine in small amounts (as it can help or hurt).\n- Maintaining a regular sleep schedule.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. **Partial Crunches**: Lie on your back with knees bent, cross arms over your chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n3. **Knee-to-Chest Stretch**: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n4. **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\n5. **Bird Dog**: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is crucial for overall health as it plays a vital role in physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults typically need 7-9 hours of sleep per night, and sleep occurs in cycles of about 90 minutes, alternating between REM (rapid eye movement) and non-REM sleep. Each stage of sleep has specific functions, such as memory consolidation during REM sleep and physical repair during deep sleep. Good sleep hygiene practices, such as maintaining a consistent sleep schedule and creating a relaxing bedtime routine, can improve sleep quality and, consequently, overall health.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\nFor Stress:\n1. Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4.\n2. Progressive muscle relaxation: Tense and release muscle groups from toes to head.\n3. Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, 1 you taste.\n4. Taking a short walk, preferably in nature.\n5. Listening to calming music.\n6. Regular exercise and adequate sleep.\n7. Mindfulness and meditation practices.\n\nFor Headaches:\n1. Drink water and stay hydrated.\n2. Apply a cold or warm compress to the head or neck.\n3. Rest in a dark, quiet room.\n4. Gentle massage of temples and neck.\n5. Use peppermint or lavender essential oils.\n6. Caffeine in small amounts (can help or hurt).\n7. Maintain a regular sleep schedule.\n\nThese remedies can help manage stress and headaches naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

*Your answer here*
It may merge more unrelated Q&A pairs. It may fail to detect topic boundaries. The semantic distance might still be small because of similar phrasing. It may produce chunks that are too large. The algorithm might want to detect question marks(?), split Q/A boundaries. Instead of relying purely on embeddings. Lower the Similarity Threshold, add more splits, chunk by Q&A boundaries.


---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [49]:
### YOUR CODE HERE
import nest_asyncio
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

# 1. Essential setup for Jupyter/Cursor
nest_asyncio.apply()

# 2. Wrap models
generator_llm = LangchainLLMWrapper(chat_model)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 3. Initialize (v0.2+ style)
generator = TestsetGenerator(
    llm=generator_llm, 
    embedding_model=generator_embeddings
)
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0)
]

# 4. Generate
# Note: Ragas now uses 'testset_size' (v0.1 used 'test_size')
dataset = generator.generate_with_langchain_docs(
    raw_docs, 
    testset_size=10, 
    query_distribution=query_distribution
)

golden_df = dataset.to_pandas()

/var/folders/pk/d_8nk69j6m50g06yn1fzvsy00000gn/T/ipykernel_5740/3927106851.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(chat_model)
/var/folders/pk/d_8nk69j6m50g06yn1fzvsy00000gn/T/ipykernel_5740/3927106851.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

In [50]:
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness
import pandas as pd
from datasets import Dataset

def evaluate_retriever(name, chain, questions, ground_truths):
    results = []
    for query, reference in zip(questions, ground_truths):
        # Run your chain
        response = chain.invoke({"question": query})
        
        # Collect data for Ragas
        results.append({
            "user_input": query,
            "response": response["response"].content,
            "retrieved_contexts": [doc.page_content for doc in response["context"]],
            "reference": reference
        })
    
    # Evaluate with Ragas
    eval_dataset = Dataset.from_list(results)
    score = evaluate(eval_dataset, metrics=[context_precision, context_recall, faithfulness])
    return score.to_pandas()

# List of your chains to evaluate
chains = {
    "Naive": naive_retrieval_chain,
    "BM25": bm25_retrieval_chain,
    "ParentDoc": parent_document_retrieval_chain,
    "MultiQuery": multi_query_retrieval_chain,
    "Ensemble": ensemble_retrieval_chain
}

all_scores = {}
for name, chain in chains.items():
    print(f"Evaluating {name}...")
    all_scores[name] = evaluate_retriever(name, chain, golden_df["user_input"], golden_df["reference"])

"""Semantic Chunking is not considered a retriever method and will not be required for marks, 
but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them?
-----
For each retriever, version A : normal chunking meaning fixed size. 
version B : Semantic chunking meaning LLM based/ meaning - based splitting. Then we evaluate both using RAGAS  : 
(Semantic chunking off ) context_precision  :  more irrelevant overlap 
(Semantic chunking off ) context_recall : sometimes lower
(Semantic chunking off ) faithfulness : fragmented context. 
(Semantic chunking on) context_precision  : high 
(Semantic chunking on) context_recall : stays high
(Semantic chunking on) faithfulness : stays more stable. 
Because without semantic chunking sentences split in the mid, retrieval may return with no proper meaning, more hallucination. 
With Semantic chunking chunks are meaningful, retriever returns cleaner context, LLM answers are more grounded.
-------
Compile these in a list and write a small paragraph about which is best for this particular data and why.
---
List of Methods used: Naive Retrieval, BM25, Parent Document Retriever, MultiQuery Retriever, Ensemble Retriever
Which is better, 
Naive :  Baseline, weakest
BM25 :  Good for keyword-heavy data, 
ParentDoc : Strong for long structured documents
MultiQuery : Better recall, more expensive
Ensemble : Usually most balanced
"""

/var/folders/pk/d_8nk69j6m50g06yn1fzvsy00000gn/T/ipykernel_5740/192301011.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness
/var/folders/pk/d_8nk69j6m50g06yn1fzvsy00000gn/T/ipykernel_5740/192301011.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness
/var/folders/pk/d_8nk69j6m50g06yn1fzvsy00000gn/T/ipykernel_5740/192301011.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: 

Evaluating Naive...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

Evaluating BM25...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

...

Evaluating MultiQuery...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

Evaluating Ensemble...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

'Semantic Chunking is not considered a retriever method and will not be required for marks, \nbut you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them?\n-----\nFor each retriever, version A : normal chunking meaning fixed size. \nversion B : Semantic chunking meaning LLM based/ meaning - based splitting. Then we evaluate both using RAGAS  : \n(Semantic chunking off ) context_precision  :  more irrelevant overlap \n(Semantic chunking off ) context_recall : sometimes lower\n(Semantic chunking off ) faithfulness : fragmented context. \n(Semantic chunking on) context_precision  : high \n(Semantic chunking on) context_recall : stays high\n(Semantic chunking on) faithfulness : stays more stable. \nBecause without semantic chunking sentences split in the mid, retrieval may return with no proper meaning, more hallucination. \nWith Semantic chunking chunks are meaningful, retriever returns cleaner context, LLM answers are more grounded.\n----